In [1]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
from glob import glob
import json
from pprint import pprint

from joblib import Parallel, delayed
from tqdm.auto import tqdm

from pysolotools.consumers import Solo

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

# graph package
import networkx as nx
from networkx.algorithms import isomorphism, bipartite

# import torch
import torch
import torch.nn.functional as F
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'torch device: {device}')
# GNN package
import torch_geometric.nn as pyg
from torch_geometric.data import Data, Dataset, InMemoryDataset, download_url
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, EdgeCNN

torch device: mps


In [3]:
# Profiling functions
# !pip install line_profiler
%load_ext line_profiler

## Process (one sample)

In [1309]:
MAP_SOLO_NAME = 'map_rooms_9'
MAP_DIR = './tmp/map'
MAP_PATH = f'{MAP_DIR}/{MAP_SOLO_NAME}'

QRY_SOLO_NAME = 'map_23'
QRY_DIR = './tmp/query-360'
QRY_PATH = f'{QRY_DIR}/{QRY_SOLO_NAME}'

In [1310]:
map_df = pd.read_csv(f'{MAP_PATH}/0.csv')
map_ans = json.load(open(f'{MAP_PATH}/5.json', 'r'))
qry_df = pd.read_csv(f'{QRY_PATH}/0.csv')
qry_ans = json.load(open(f'{QRY_PATH}/0.json', 'r'))

In [1311]:
map_df['node_id'] = [f'map_{idx}' for idx in map_df.index]
qry_df['node_id'] = [f'qry_{idx}' for idx in qry_df.index]

In [1312]:
print('len: ', len(map_df))
map_df.head()

len:  514


,instanceId,labelName,distToCam,isBlockToCam,abs_x,abs_y,abs_z,rel_x,rel_y,rel_z,sph_r,sph_elevation,sph_azimuth,node_id
0,1,storage,12.909491,True,-15.745,6.1000,-12.930,9.245,-1.2000,8.930,12.909490,-0.093089,0.768068,map_0
1,2,cup,4.624875,False,-5.389,6.4644,0.411,-1.111,-0.8356,-4.411,4.624875,-0.181673,-1.817535,map_1
2,3,storage,3.763566,False,-3.073,6.1000,-3.010,-3.427,-1.2000,-0.990,3.763566,-0.324512,-2.860366,map_2
3,4,poster,4.548761,False,-3.049,7.9050,-1.099,-3.451,0.6050,-2.901,4.548761,0.133398,-2.442566,map_3
4,5,plant,4.849875,False,-8.342,6.1000,0.323,1.842,-1.2000,-4.323,4.849876,-0.250026,-1.168000,map_4


In [1313]:
print('len: ', len(qry_df))
qry_df.head()

len:  514


,instanceId,labelName,distToCam,isBlockToCam,abs_x,abs_y,abs_z,rel_x,rel_y,rel_z,sph_r,sph_elevation,sph_azimuth,node_id
0,1,storage,9.108857,True,-15.745,6.1000,-12.930,-8.253272,-1.2000,-3.662619,9.108857,-0.132124,-2.723925,qry_0
1,2,cup,25.221544,True,-5.389,6.4644,0.411,-18.609272,-0.8356,-17.003618,25.221544,-0.033136,-2.401250,qry_1
2,3,storage,24.975880,True,-3.073,6.1000,-3.010,-20.925272,-1.2000,-13.582619,24.975879,-0.048065,-2.565850,qry_2
3,4,poster,26.063198,True,-3.049,7.9050,-1.099,-20.949272,0.6050,-15.493619,26.063197,0.023215,-2.504795,qry_3
4,5,plant,23.080230,True,-8.342,6.1000,0.323,-15.656272,-1.2000,-16.915619,23.080230,-0.052016,-2.317550,qry_4


In [1314]:
# filter qry and map nodes

qry_df = qry_df[qry_df['isBlockToCam'] == False]
map_df = map_df[map_df['isBlockToCam'] == False]

intersect_labels = np.intersect1d(map_df['labelName'].unique(), qry_df['labelName'].unique())

submap_df = map_df.loc[
    map_df['labelName'].isin(intersect_labels), 
    ['node_id', 'labelName', 'sph_azimuth']
]
subqry_df = qry_df.loc[
    qry_df['labelName'].isin(intersect_labels), 
    ['node_id', 'labelName', 'sph_azimuth']
]

In [1315]:
print('len: ', len(submap_df))
submap_df.head()

len:  182


,node_id,labelName,sph_azimuth
1,map_1,cup,-1.817535
2,map_2,storage,-2.860366
3,map_3,poster,-2.442566
5,map_5,chair,1.740465
6,map_6,chair,0.315803


In [1316]:
print('len: ', len(subqry_df))
subqry_df.head()

len:  68


,node_id,labelName,sph_azimuth
11,qry_11,louver,1.466901
12,qry_12,paper,-2.274879
56,qry_56,louver,-0.165420
123,qry_123,storage,-1.999515
124,qry_124,storage,-2.171888


In [1317]:
submap_df = submap_df.sort_values('labelName')
subqry_df = subqry_df.sort_values('labelName')
join_df = submap_df.join(subqry_df.set_index('labelName'), on='labelName', lsuffix='_map', rsuffix='_qry').reset_index(drop=True)

In [1318]:
print('len: ', len(join_df))
join_df

len:  1731


,node_id_map,labelName,sph_azimuth_map,node_id_qry,sph_azimuth_qry
0,map_135,book,0.116224,qry_311,1.321305
1,map_135,book,0.116224,qry_314,1.345607
2,map_135,book,0.116224,qry_288,-0.531084
3,map_135,book,0.116224,qry_287,-0.568421
4,map_135,book,0.116224,qry_286,-0.583472
...,...,...,...,...,...
1726,map_152,table_lamp,0.265590,qry_376,3.033416
1727,map_452,table_lamp,-0.230746,qry_376,3.033416
1728,map_451,table_lamp,-0.343621,qry_376,3.033416
1729,map_151,table_lamp,0.135551,qry_376,3.033416


In [1319]:
anchor_idx = 0
anchor = join_df.loc[anchor_idx]
rot_map = anchor['sph_azimuth_qry'] - anchor['sph_azimuth_map']

aligned_azi_map = ((join_df['sph_azimuth_map'] + rot_map) + np.pi) % (2*np.pi) - np.pi
dis = aligned_azi_map - join_df['sph_azimuth_qry']
weight = np.minimum(np.abs(dis), np.abs(dis % (2*np.pi)))

join_df['weight'] = weight
join_df['weight']

0       0.000000
1       0.024302
2       1.852389
3       1.889726
4       1.904777
          ...   
1726    1.562745
1727    2.059081
1728    2.171956
1729    1.692784
1730    0.682828
Name: weight, Length: 1731, dtype: float64

In [1647]:
node_ids_map = submap_df['node_id'].tolist()
node_ids_qry = subqry_df['node_id'].tolist()
node_ids_trash = [f'trash_{id_qry}' for id_qry in node_ids_qry]

G = nx.from_pandas_edgelist(join_df, 'node_id_map', 'node_id_qry', ['weight'])

# build trash nodes
G.add_nodes_from(node_ids_trash)
for node_id_qry, node_id_trash in zip(node_ids_qry, node_ids_trash):
    G.add_edge(node_id_trash, node_id_qry, weight=1000)

min_matching = bipartite.minimum_weight_full_matching(G, top_nodes=node_ids_qry)

cost = sum(G.edges[u, v]['weight'] for u, v in min_matching.items()) / 2

# min_matching, cost

In [1648]:
best_matching_angle = 0.
best_matching_cost = np.inf

# rotate qry by following angles
# rotation_angles = join_df['sph_azimuth_map'] - join_df['sph_azimuth_qry']  # align every query object with every map object with the same category
# if len(rotation_angles) > 360:
rotation_angles = np.linspace(-np.pi, np.pi, 16)  # every 1 degree

node_ids_map = submap_df['node_id'].tolist()
node_ids_qry = subqry_df['node_id'].tolist()
node_ids_trash = [f'trash_{id_qry}' for id_qry in node_ids_qry]

for rotation_angle in tqdm(rotation_angles):
    aligned_azi_map = ((join_df['sph_azimuth_map'] + rotation_angle) + np.pi) % (2*np.pi) - np.pi
    dis = aligned_azi_map - join_df['sph_azimuth_qry']
    weight = np.minimum(np.abs(dis), np.abs(dis % (2*np.pi)))
    join_df['weight'] = weight

    # subjoin_df = join_df[join_df['weight'] < qry_ans['FOV']]
    # G = nx.from_pandas_edgelist(subjoin_df, 'node_id_map', 'node_id_qry', ['weight'])
    
    G = nx.from_pandas_edgelist(join_df, 'node_id_map', 'node_id_qry', ['weight'])
    # build trash nodes
    G.add_nodes_from(node_ids_trash)
    for node_id_qry, node_id_trash in zip(node_ids_qry, node_ids_trash):
        G.add_edge(node_id_trash, node_id_qry, weight=1000)
    
    min_matching = bipartite.minimum_weight_full_matching(G, top_nodes=node_ids_qry)

    cost = sum(G.edges[u, v]['weight'] for u, v in min_matching.items()) / 2

    # non_trash_matching = [(u, v) for u, v in min_matching.items() if u in node_ids_qry and v in node_ids_map]
    # trash_matching = [(u, v) for u, v in min_matching.items() if u in node_ids_qry and v in node_ids_trash]

    # cost = sum(G.edges[u, v]['weight'] for u, v in min_matching.items() if u in node_ids_qry and v in node_ids_map)
    # cost += len(trash_matching) * np.pi
    # cost /= len(node_ids_qry)
    # cost *= len(node_ids_qry) / len(non_trash_matching)
    # cost /= len(non_trash_matching)
    
    if cost < best_matching_cost:
        best_matching_cost = cost
        best_matching_angle = rotation_angle

print()
print(f'[end] rotation angle: {best_matching_angle:.3f}, loss: {best_matching_cost:.3f}')

  0%|          | 0/16 [00:00<?, ?it/s]


[end] rotation angle: -1.466, loss: 61.864


In [1649]:
def rotate(df, rot_angle):
    return ((df + rot_angle) + np.pi) % (2*np.pi) - np.pi

In [1650]:
vis_map = rotate(submap_df['sph_azimuth'], best_matching_angle)
vis_map = pd.concat((vis_map, submap_df['labelName']), axis=1)
# vis_map

In [1651]:
fig = vis_azimuth(vis_map, 'sph_azimuth', 'labelName', title='map')
fig.show()

In [1652]:
fig = vis_azimuth(subqry_df, 'sph_azimuth', 'labelName', title='qry')
fig.show()

## Process (all samples)

In [41]:
MAP_SOLO_NAME = 'all360_68'
MAP_DIR = './graph/qry360'
MAP_PATH = f'{MAP_DIR}/{MAP_SOLO_NAME}'

QRY_SOLO_NAME = 'all360_179'
QRY_DIR = './graph/qry360'
QRY_PATH = f'{QRY_DIR}/{QRY_SOLO_NAME}'

In [42]:
n_maps = len(glob(f'{MAP_PATH}/*.csv'))
n_qrys = len(glob(f'{QRY_PATH}/*.csv'))
print(f'num of maps: {n_maps}')
print(f'num of qrys: {n_qrys}')

map_dfs = [pd.read_csv(f'{MAP_PATH}/{i}.csv', index_col='instanceId') for i in range(n_maps)]
map_ans_df = pd.DataFrame([json.load(open(f'{MAP_PATH}/{i}.json', 'r')) for i in range(n_maps)])

qry_dfs = [pd.read_csv(f'{QRY_PATH}/{i}.csv', index_col='instanceId') for i in range(n_qrys) if os.path.exists(f'{QRY_PATH}/{i}.csv')]
qry_ans_df = pd.DataFrame([json.load(open(f'{QRY_PATH}/{i}.json', 'r')) for i in range(n_qrys) if os.path.exists(f'{QRY_PATH}/{i}.json')])

qry_ans_df['closest_match'] = qry_ans_df.T.apply(lambda c: ((c[['pos_x', 'pos_y', 'pos_z']] - map_ans_df[['pos_x', 'pos_y', 'pos_z']])**2).sum(axis=1).pow(1./2).astype(float).argmin())
qry_ans_df['closest_match'].value_counts()

num of maps: 68
num of qrys: 179


closest_match
51    5
29    5
43    4
19    4
38    4
     ..
13    1
12    1
52    1
6     1
48    1
Name: count, Length: 68, dtype: int64

In [43]:
def prep_data_pairs(map_dfs, qry_dfs):
    res = []

    for map_df in map_dfs:
        map_df['node_id'] = [f'map_{idx}' for idx in map_df.index]
    for qry_df in qry_dfs:
        qry_df['node_id'] = [f'qry_{idx}' for idx in qry_df.index]
    
    for qry_idx, qry_df in enumerate(qry_dfs):
        for map_idx, map_df in enumerate(map_dfs):

            qry_df = qry_df
            # qry_df = qry_df[qry_df['sph_azimuth'] > 0]
            # qry_df = qry_df[qry_df['sph_azimuth'] < np.deg2rad(60)]
            # if 'isBlockToCam' in qry_df.columns:
                # qry_df = qry_df[qry_df['isBlockToCam'] == False]
            # qry_df = qry_df[np.all((qry_df['distToCam'] >= 0.1, qry_df['distToCam'] <= 5), axis=0)]
            # qry_df = qry_df.sort_values('sph_r').iloc[:20]

            map_df = map_df
            # map_df = map_df[map_df['isBlockToCam'] == False]


            intersect_labels = np.intersect1d(map_df['labelName'].unique(), qry_df['labelName'].unique(), assume_unique=True)

            submap_df = map_df.loc[
                map_df['labelName'].isin(intersect_labels), 
                ['node_id', 'labelName', 'sph_azimuth']
            ]
            subqry_df = qry_df.loc[
                qry_df['labelName'].isin(intersect_labels), 
                ['node_id', 'labelName', 'sph_azimuth']
            ]

            submap_df = submap_df.sort_values('labelName')
            subqry_df = subqry_df.sort_values('labelName')
            join_df = submap_df.join(subqry_df.set_index('labelName'), on='labelName', lsuffix='_map', rsuffix='_qry').reset_index(drop=True)

            res.append({
                'map_idx': map_idx, 
                'qry_idx': qry_idx, 
                'map_df': map_df,
                'qry_df': qry_df,
                'submap_df': submap_df,
                'subqry_df': subqry_df,
                'join_df': join_df
            })

    return pd.DataFrame(res)

In [84]:
def calc_best_match(map_idx, qry_idx, map_df, qry_df, submap_df, subqry_df, join_df, max_n_try=16, verbose=True):
    best_matching = None
    best_matching_angle = 0.
    best_matching_cost = np.inf
    
    rotation_angles = join_df['sph_azimuth_map'] - join_df['sph_azimuth_qry']  # align every query object with every map object with the same category
    if len(rotation_angles) > max_n_try:
        # rotation_angles = np.linspace(-np.pi, np.pi, max_n_try, endpoint=False)  # equally distributed
        rotation_angles = np.linspace(np.min(rotation_angles), np.max(rotation_angles), max_n_try)
    
    node_ids_map = submap_df['node_id'].tolist()
    node_ids_qry = subqry_df['node_id'].tolist()
    node_ids_trash = [f'trash_{id_qry}' for id_qry in node_ids_qry]

    # manually defined weight
    trash_matching_weight = 1000
    
    for rotation_angle in rotation_angles:
        aligned_azi_map = ((join_df['sph_azimuth_map'] + rotation_angle) + np.pi) % (2*np.pi) - np.pi
        dis = aligned_azi_map - join_df['sph_azimuth_qry']
        weight = np.minimum(np.abs(dis), np.abs(dis % (2*np.pi)))
        join_df['weight'] = weight

        G = nx.from_pandas_edgelist(join_df, 'node_id_map', 'node_id_qry', ['weight'])
        # build trash nodes
        G.add_nodes_from(node_ids_trash)
        for node_id_qry, node_id_trash in zip(node_ids_qry, node_ids_trash):
            G.add_edge(node_id_trash, node_id_qry, weight=trash_matching_weight)
        
        min_matching = bipartite.minimum_weight_full_matching(G, top_nodes=node_ids_qry)
        
        # cost = sum(G.edges[u, v]['weight'] for u, v in min_matching.items()) / 2
    
        # non_trash_matching = [(u, v) for u, v in min_matching.items() if u in node_ids_qry and v in node_ids_map]
        trash_matching = [(u, v) for u, v in min_matching.items() if u in node_ids_qry and v in node_ids_trash]

        cost = sum(G.edges[u, v]['weight'] for u, v in min_matching.items() if u in node_ids_qry and v in node_ids_map)
        # cost += len(trash_matching) * np.pi * max(len(node_ids_qry)/len(node_ids_map), len(node_ids_map)/len(node_ids_qry))
        cost += len(trash_matching) * np.pi
        # cost += (len(qry_df) - len(subqry_df)) * np.pi
        cost /= len(node_ids_qry)
        
        if cost < best_matching_cost:
            best_matching = min_matching
            best_matching_cost = cost
            best_matching_angle = rotation_angle
            # print(f'[update] rotation angle: {best_matching_angle:.3f}, loss: {best_matching_cost:.3f}')

    if verbose:
        print(f'[qry-{qry_idx} to map-{map_idx}] rotation angle: {best_matching_angle:.3f}, loss: {best_matching_cost:.3f}')
            
    return best_matching_angle, best_matching_cost

In [85]:
def process(data_pair):
    map_idx = data_pair['map_idx']
    qry_idx = data_pair['qry_idx']
    # map_df = data_pair['map_df']
    # qry_df = data_pair['qry_df']
    # join_df = data_pair['join_df']
    submap_df = data_pair['submap_df']
    subqry_df = data_pair['subqry_df']
    
    angle, cost = calc_best_match(**data_pair, max_n_try=16, verbose=False)
    
    match = {
        'qry_idx': qry_idx,
        'map_idx': map_idx,
        'loss': cost,
        'angle': angle,
        'submap_len': len(submap_df),
        'subqry_len': len(subqry_df),
    }
    
    return match

data_pairs = pd.DataFrame(prep_data_pairs(map_dfs, qry_dfs))
results = Parallel(n_jobs=-1)(delayed(process)(data_pair) for _, data_pair in tqdm(data_pairs.iterrows(), total=len(data_pairs)))
matches_df = pd.DataFrame(results).astype({'qry_idx': 'int', 'map_idx': 'int'})

  0%|          | 0/12172 [00:00<?, ?it/s]

In [86]:
# matches = []

# data_pairs = pd.DataFrame(prep_data_pairs(map_dfs, qry_dfs))

# for _, data_pair in tqdm(data_pairs.iterrows(), total=len(data_pairs)):
#     map_idx = data_pair['map_idx']
#     qry_idx = data_pair['qry_idx']
#     # map_df = data_pair['map_df']
#     # qry_df = data_pair['qry_df']
#     # join_df = data_pair['join_df']
#     submap_df = data_pair['submap_df']
#     subqry_df = data_pair['subqry_df']

#     angle, cost = calc_best_match(**data_pair, max_n_try=16, verbose=False)

#     match = {
#         'qry_idx': qry_idx,
#         'map_idx': map_idx,
#         'loss': cost,
#         'angle': angle,
#         'submap_len': len(submap_df),
#         'subqry_len': len(subqry_df),
#     }
#     matches.append(match)
#     # print(f'[update] qry: {qry_idx}, map: {map_idx}, loss: {loss:.3f}')

# matches_df = pd.DataFrame(matches).astype({'qry_idx': 'int', 'map_idx': 'int'})

In [87]:
matches_df

,qry_idx,map_idx,loss,angle,submap_len,subqry_len
0,0,0,0.138161,-0.583126,170,165
1,0,1,0.604025,-4.530877,141,164
2,0,2,1.621523,5.199454,89,162
3,0,3,1.750542,4.474849,81,162
4,0,4,1.434195,-1.348757,100,162
...,...,...,...,...,...,...
12167,178,63,0.655576,-2.238299,156,151
12168,178,64,0.590315,-2.931375,175,151
12169,178,65,0.636558,-4.574513,152,151
12170,178,66,0.495748,1.234504,173,151


In [88]:
loss_tbl = matches_df.pivot(index='qry_idx', columns=['map_idx'], values=['loss'])
loss_tbl

loss                                                              \
map_idx        0         1         2         3         4         5         6    
qry_idx                                                                         
0        0.138161  0.604025  1.621523  1.750542  1.434195  1.533898  1.559512   
1        0.167037  0.428269  1.498296  1.662664  1.332736  1.407345  1.468653   
2        0.336430  0.381052  1.126452  1.336621  0.861180  1.102708  1.067499   
3        0.185667  0.316387  1.289514  1.445071  1.089819  1.258043  1.213935   
4        0.207929  0.174917  0.848119  1.101284  0.652125  0.791659  0.867876   
...           ...       ...       ...       ...       ...       ...       ...   
174      0.462301  0.652019  1.451282  1.532064  1.340241  1.472109  1.395339   
175      0.481374  0.712296  1.526301  1.673168  1.414275  1.532605  1.551670   
176      0.350851  0.704890  1.442204  1.486062  1.273612  1.410091  1.264178   
177      0.148251  0.637834  1.564860  1.655738  1.387682  1.490635  1.429996   
178      0.571724  0.693228  1.333225  1.485824  1.174505  1.462534  1.377359   

                                       ...                                \
map_idx        7         8         9   ...        58        59        60   
qry_idx                                ...                                 
0        1.616898  1.756727  1.253037  ...  0.565626  2.084911  0.944256   
1        1.524136  1.663073  1.158374  ...  0.404168  2.008340  0.780937   
2        1.186692  1.342953  0.844801  ...  0.274498  1.786626  0.667659   
3        1.357734  1.498578  0.979060  ...  0.313175  1.875953  0.664025   
4        0.907853  1.112473  0.588606  ...  0.178510  1.572729  0.473167   
...           ...       ...       ...  ...       ...       ...       ...   
174      1.601306  1.674961  1.205383  ...  0.472026  2.039282  0.674631   
175      1.625002  1.760513  1.190713  ...  0.513176  2.088574  0.879937   
176      1.581252  1.567809  1.189541  ...  0.538830  1.926364  0.674349   
177      1.556800  1.614038  1.270804  ...  0.533841  2.068463  0.773169   
178      1.538807  1.578464  1.137028  ...  0.303010  1.918248  0.411134   

                                                                               
map_idx        61        62        63        64        65        66        67  
qry_idx                                                                        
0        1.194457  0.787147  0.869304  0.678538  0.462907  0.445841  0.695546  
1        1.063526  0.620101  0.675140  0.530752  0.402228  0.307757  0.622741  
2        0.651612  0.370844  0.564800  0.332582  0.518189  0.302129  0.683145  
3        0.791258  0.629323  0.672984  0.484948  0.240495  0.267231  0.492034  
4        0.498928  0.453945  0.443785  0.228100  0.290676  0.208492  0.525910  
...           ...       ...       ...       ...       ...       ...       ...  
174      1.099572  0.543496  0.726047  0.631927  0.514599  0.373626  0.229140  
175      1.083015  0.718804  0.718023  0.583083  0.603926  0.300396  0.538268  
176      1.014839  0.467963  0.785165  0.755188  0.416259  0.420575  0.339356  
177      1.100070  0.634293  0.761701  0.788780  0.396130  0.424493  0.515301  
178      0.963005  0.216322  0.655576  0.590315  0.636558  0.495748  0.590577  

[179 rows x 68 columns]

In [89]:
(loss_tbl.loc[(loss_tbl == np.inf).sum(axis=1) != 0] == np.inf).sum(axis=1)

Series([], dtype: float64)

In [90]:
plt.figure(figsize=(4,3))
(loss_tbl < 1000).sum(axis=1).value_counts().sort_index().cumsum()#.plot()

68    179
Name: count, dtype: int64

<Figure size 400x300 with 0 Axes>

In [91]:
min_loss = pd.DataFrame(np.argsort(loss_tbl, axis=1), index=loss_tbl.index)
min_loss

,0,1,2,3,4,5,6,7,8,9,...,58,59,60,61,62,63,64,65,66,67
qry_idx,,,,,,,,,,,,,,,,,,,,,
0,0,38,42,66,65,39,53,40,51,41,...,7,2,28,27,3,8,25,24,16,59
1,0,38,66,53,51,65,58,42,39,1,...,2,7,28,3,8,27,25,24,16,59
2,55,42,38,53,39,41,58,66,40,64,...,17,7,28,3,8,27,25,24,59,16
3,0,38,65,53,39,66,51,55,58,1,...,47,7,28,3,27,8,25,24,16,59
4,42,53,55,1,38,58,51,0,66,46,...,7,17,28,3,8,27,25,24,59,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174,67,39,38,66,41,51,42,40,0,55,...,5,3,7,8,28,27,25,24,16,59
175,66,38,39,42,53,51,41,0,55,58,...,47,7,28,3,27,25,8,24,16,59
176,39,37,38,67,0,40,65,66,42,41,...,2,3,8,28,27,7,16,25,24,59


In [92]:
# map_ans_df['roomName'].to_dict()

In [93]:
min_loss_room = min_loss.applymap(lambda v: map_ans_df['roomName'].to_dict()[v])
# min_loss_room

In [98]:
ans_map_ids = qry_ans_df['closest_match']
# ans_map_ids = qry_ans_df['roomName']
# print('ans idx:', ans_map_ids.tolist())

In [99]:
rank_diff = pd.Series([np.argwhere(min_loss.T[i] == ans_map_ids[i])[0][0] for i in range(len(ans_map_ids))], index=min_loss.index)
# rank_diff = pd.Series([np.argwhere(min_loss_room.T[i] == ans_map_ids[i])[0][0] for i in range(len(ans_map_ids))], index=min_loss_room.index)

print(f'rank diff: mean {np.mean(rank_diff):.3f}, std {np.std(rank_diff):.3f}, median {np.median(rank_diff):.3f}')
print('rank diff:', rank_diff.values)

rank diff: mean 14.966, std 18.139, median 7.000
rank diff: [ 0  9 15  9 54 33 30 52 59 42 26  4 39 21 22 33  2 39 44 58 50 49 37 31
 19 46 47  8 41 23  5 29 30 11 14 14  1 13  0 33 64 64  1  3 45  5  5  4
 39  1 33 11 16 22  9 12 33  5 15 33 51 50 17 48 13  7  0  7 42  7 42  7
 41  0  9 12 17 11 26 17  2  9  2  1 17  0  2 22 39 11 14  6 18  0  5 21
  1  0  1  0  5  1  2  0  0  3  1  0  1  0  1  0  4  9  2  6  0  8  5  8
  1  8  2 13 12 15 19  9 50 58  0 19  0  0  0  0  2  0  3  0  1  7 28  0
  0  0  0  0  1  0  0  0 66 67 66  6  1 11  0  6  3  8  1  5  6  0  0  1
  0  0  2  0  0  0  0 11  6  2  0]


In [100]:
# pd.concat({'k': (loss_tbl < 1000).sum(axis=1), 'diff': pd.Series(rank_diff)}, axis=1).groupby('k').agg(['mean', 'count', 'min', 'max'])

In [101]:
rk_count = pd.DataFrame(rank_diff).value_counts()
for i in range(n_maps):
    if i not in rk_count.index:
        rk_count[i] = 0
rk_count = rk_count.sort_index()
rk_ratio = rk_count.cumsum() / rk_count.sum()

print(f'R@1: {rk_count[:1].sum()}/{rk_count.sum()} {rk_ratio[1-1] * 100 :.2f}%')
print(f'R@3: {rk_count[:3].sum()}/{rk_count.sum()} {rk_ratio[3-1] * 100 :.2f}%')
print(f'R@5: {rk_count[:5].sum()}/{rk_count.sum()} {rk_ratio[5-1] * 100 :.2f}%')
rk_count

R@1: 39/179 21.79%
R@3: 65/179 36.31%
R@5: 72/179 40.22%


0     39
1     16
2     10
3      4
4      3
      ..
63     0
64     2
65     0
66     2
67     1
Name: count, Length: 68, dtype: int64

In [59]:
# min_loss[0].value_counts()

In [60]:
map_ans_df['src'] = 'map'
qry_ans_df['src'] = 'qry'
map_ans_df['idx'] = map_ans_df['src'] + '_' + map_ans_df.index.astype(str).to_numpy()
qry_ans_df['idx'] = qry_ans_df['src'] + '_' + qry_ans_df.index.astype(str).to_numpy()
qry_ans_df['match'] = min_loss[0]
qry_ans_df['match_room'] = min_loss_room[0]

vis_3d_df = pd.concat((map_ans_df, qry_ans_df), axis=0).reset_index(drop=True)
vis_3d_df = vis_3d_df.astype({'match': 'str'})
vis_3d_df = vis_3d_df.astype({'roomName': 'str', 'match_room': 'str'})

In [61]:
map_ans_df.head()

,pos_x,pos_y,pos_z,visibleObjectInstanceIds,roomName,src,idx
0,2.168203,1.7,-9.366982,"[26, 57, 80, 81, 91, 95, 98, 100, 101, 112, 11...",office (RB),map,map_0
1,4.637363,1.7,-4.759018,"[26, 48, 57, 80, 81, 91, 95, 101, 112, 114, 11...",office (RB),map,map_1
2,1.481754,1.7,-0.251942,"[26, 46, 57, 60, 126, 141, 160, 163, 180, 229,...",office (RB),map,map_2
3,9.284369,1.7,-0.310085,"[12, 26, 57, 72, 81, 106, 120, 141, 143, 160, ...",office (RB),map,map_3
4,7.930358,1.7,4.661423,"[26, 46, 57, 80, 81, 97, 106, 141, 160, 180, 2...",office (RB),map,map_4


In [62]:
fig = px.scatter(vis_3d_df, x='pos_x', y='pos_z', color='match_room', symbol='src', hover_name='idx', hover_data='roomName')
fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1,
)
fig.show()

min_loss.loc[:, 0:0].T

qry_idx,0,1,2,3,4,5,6,7,8,9,...,169,170,171,172,173,174,175,176,177,178
0,0,0,55,0,42,53,38,39,55,53,...,66,55,0,0,62,67,66,39,0,62


In [166]:
# matches_df.groupby('map_idx')[['submap_len', 'subqry_len']].agg(['mean', 'std', 'median'])

In [24]:
matches_df.groupby('map_idx')['loss'].agg(lambda x: np.mean(x.values, where=(x!=np.inf)))

map_idx
0     1.256679
1     0.895221
2     1.007897
3     0.879129
4     1.296695
5     1.308062
6     1.033340
7     1.201352
8     1.134186
9     1.319476
10    1.250817
11    1.304857
12    1.038117
13    1.029531
14    1.070803
15    0.992352
16    1.276857
17    1.491354
18    1.481007
19    1.353257
20    1.342914
21    1.776157
22    1.408174
23    1.076593
24    1.806545
25    1.834080
Name: loss, dtype: float64

In [25]:
px.scatter( 
    pd.concat((matches_df.groupby('map_idx')['loss'].agg(lambda x: np.mean(x.values, where=(x!=np.inf))), n_visible_objs), axis=1).reset_index(names=['map_idx']),
    x='n_visible_objs', 
    y='loss',
    color='map_idx',
)

## Visualization

In [3]:
def vis_azimuth(df, data_col, name_col, title='azimuth'):
    vis_df = pd.DataFrame({'vis_x': np.cos(df[data_col]), 'vis_y': np.sin(df[data_col]), 'labelName': df[name_col]}, index=df.index)
    fig = px.scatter(vis_df, x='vis_x', y='vis_y', color='labelName')
    fig.update_layout(
        title=title,
        autosize=False,
        width=400,
        height=400,
    )
    fig.update_xaxes(range=[-1.2, 1.2])
    fig.update_yaxes(range=[-1.2, 1.2])
    return fig

In [26]:
fig = vis_azimuth(matches_df[2]['match_df'], 'rotated_azi', 'labelName', title='qry')
fig.show()

NameError: name 'vis_azimuth' is not defined

In [27]:
fig = vis_azimuth(matches_df[2]['match_df'], 'sph_azimuth_map', 'labelName', title='map')
fig.show()

NameError: name 'vis_azimuth' is not defined